# broadcast-initial-weights — worked example 2: Broadcast the full state_dict so BatchNorm buffers sync too

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcast-initial-weights`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`model.parameters()` only yields learnable tensors — it SKIPS buffers like BatchNorm's `running_mean` / `running_var`. To make replicas truly identical you must broadcast every tensor in `model.state_dict().values()`, which includes params AND buffers. The iteration order is stable across ranks because every rank shares the same module graph.

## Worked solution

**Goal.** Demonstrate that syncing only `parameters()` leaves BN buffers divergent, and that iterating `state_dict().values()` fixes it.

1. **A model with buffers.** `BatchNorm1d` registers `running_mean` and `running_var` as buffers, not parameters. We poke them to rank-specific values to mimic divergent state.
2. **The trap.** A `parameters()`-only loop would never touch `running_mean`, so rank 1's buffer would stay wrong — silently corrupting eval-mode behaviour.
3. **The fix.** Loop over `state_dict().values()` and broadcast each tensor. The state dict packs weights, biases, AND buffers in a deterministic order, so tensor i on rank 1 lines up with tensor i on rank 0.
4. **In-place copy.** `state_dict()` returns references to the live tensors (not copies), so `copy_` into them mutates the module's actual storage — no reload required.
5. **Check.** After the loop, rank 1's `running_mean` equals rank 0's, and we report the number of tensors visited (should exceed the parameter count, proving buffers were included).

In [ ]:
class FakeDist:
    def __init__(self, rank0_tensors):
        self.rank0_tensors = rank0_tensors
        self._i = 0
    def reset(self):
        self._i = 0
    def broadcast(self, tensor, src=0):
        tensor.copy_(self.rank0_tensors[self._i])
        self._i += 1

def build_model(rank):
    m = t.nn.Sequential(t.nn.Linear(3, 3, bias=False), t.nn.BatchNorm1d(3))
    with t.no_grad():
        m[0].weight.fill_(float(rank + 1))
        m[1].running_mean.fill_(float(10 * (rank + 1)))
    return m

def broadcast_state(model, fake):
    fake.reset()
    count = 0
    for tensor in model.state_dict().values():
        fake.broadcast(tensor, src=0)
        count += 1
    return count

rank0 = build_model(0)
fake = FakeDist([v.clone() for v in rank0.state_dict().values()])

rank1 = build_model(1)
n_params = sum(1 for _ in rank1.parameters())
print('rank1 running_mean before:', rank1.state_dict()['1.running_mean'].tolist())
n_broadcast = broadcast_state(rank1, fake)
print('rank1 running_mean after: ', rank1.state_dict()['1.running_mean'].tolist())
print('tensors broadcast:', n_broadcast, '| param tensors:', n_params)
print('buffers were included:', n_broadcast > n_params)